# Deduplication Impact Experiment

**Research Question:** Does near-duplicate removal improve re-identification quality, 
or does it merely reduce training data volume?

This notebook compares MiewID-MSv2 trained with ArcFace on:
1. **Deduplicated dataset** (`JID_HF_0226_Segmented_Deduplicated_Cached`) — 1,998 images, 175 identities
2. **Non-deduplicated dataset** (`JID_HF_0226_Segmented_Cached`) — segmented but with duplicates retained

**Fixed Settings:**
- Backbone: MiewID-MSv2 (EfficientNetV2, 2152-dim, input 440)
- Loss: ArcFace (m=0.5, s=64)
- Epochs: 50, AdamW (lr=1e-4, wd=1e-4), seed 42
- Evaluation: mAP, CMC@k, identity-balanced mAP, mAP@9+

Results tracked in W&B project: `camera-trap-reidentification`, tags: `dedup_comparison`

## Step 1: Create Non-Deduplicated Dataset Variant

Export a segmented-but-not-deduplicated variant from the master dataset to FiftyOne.

In [1]:
import sys
from pathlib import Path
import logging

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import fiftyone as fo
from fiftyone import ViewField as F

from jaguars.common.logging_utils import setup_logger
from jaguars.common.config import JID_MASTER_DATASET

logger = setup_logger('dedup_experiments', level=logging.INFO)
print('Imports successful')

Imports successful


In [2]:
# Check available datasets
print('Available datasets:', fo.list_datasets())

DEDUP_DATASET = 'JID_HF_0226_Segmented_Deduplicated_Cached'
NON_DEDUP_DATASET = 'JID_HF_0226_Segmented_Cached'  # Will be created

# Check if non-deduplicated variant already exists
if NON_DEDUP_DATASET in fo.list_datasets():
    print(f'Non-deduplicated dataset already exists: {NON_DEDUP_DATASET}')
    ds_nodedup = fo.load_dataset(NON_DEDUP_DATASET)
    imgs = ds_nodedup.select_group_slices('image') if ds_nodedup.group_field else ds_nodedup
    print(f'  Images: {len(imgs)}, Identities: {len(imgs.count_values("ground_truth.label"))}')
else:
    print(f'Dataset {NON_DEDUP_DATASET} not found -- will create from master.')

Available datasets: ['JID_HF_0226_Segmented_Deduplicated_Cached', 'JID_Master_Dataset']
Dataset JID_HF_0226_Segmented_Cached not found -- will create from master.


In [3]:
# Create non-deduplicated variant from master dataset
# This keeps segmented samples but does NOT remove duplicates

if NON_DEDUP_DATASET not in fo.list_datasets():
    master = fo.load_dataset(JID_MASTER_DATASET)
    imgs_master = master.select_group_slices('image')
    
    # Apply same filters as deduplicated variant, EXCEPT deduplication
    # 1. Must have segmentation
    view = imgs_master.exists('sam3_segmentations')
    # 2. Exclude filter_count and filter_quality tagged samples
    view = view.match_tags('filter_count', bool=False)
    view = view.match_tags('filter_quality', bool=False)
    # 3. Do NOT filter duplicates (this is the key difference)
    
    print(f'Non-deduplicated view: {len(view)} samples')
    print(f'  Identities: {len(view.count_values("ground_truth.label"))}')
    print(f'  Splits: {view.count_values("closed_set_split")}')
    
    # Clone as new dataset
    ds_nodedup = view.clone(name=NON_DEDUP_DATASET)
    ds_nodedup.save()
    print(f'Created dataset: {NON_DEDUP_DATASET}')
    print(f'  Total: {len(ds_nodedup)} samples')
else:
    print(f'Dataset already exists: {NON_DEDUP_DATASET}')

Non-deduplicated view: 2738 samples
  Identities: 187
  Splits: {'test': 263, 'val': 238, 'train': 2237}
Created dataset: JID_HF_0226_Segmented_Cached
  Total: 2738 samples


In [4]:
# Compare dataset statistics
print('=' * 60)
print('Dataset Comparison')
print('=' * 60)

for name in [DEDUP_DATASET, NON_DEDUP_DATASET]:
    ds = fo.load_dataset(name)
    imgs = ds.select_group_slices('image') if ds.group_field else ds
    gt = imgs.count_values('ground_truth.label')
    counts = sorted(gt.values(), reverse=True)
    splits = imgs.count_values('closed_set_split')
    
    print(f'\n{name}:')
    print(f'  Samples: {len(imgs)}')
    print(f'  Identities: {len(gt)}')
    print(f'  Splits: {splits}')
    print(f'  Samples/identity: max={counts[0]}, median={counts[len(counts)//2]}, min={counts[-1]}')

Dataset Comparison

JID_HF_0226_Segmented_Deduplicated_Cached:
  Samples: 1998
  Identities: 175
  Splits: {'train': 1632, 'val': 167, 'test': 199}
  Samples/identity: max=139, median=4, min=1

JID_HF_0226_Segmented_Cached:
  Samples: 2738
  Identities: 187
  Splits: {'train': 2237, 'val': 238, 'test': 263}
  Samples/identity: max=161, median=5, min=1


## Step 2: (Optional) Publish Non-Deduplicated Dataset to Hugging Face

In [5]:
# Uncomment to upload to HuggingFace
# from huggingface_hub import HfApi, login
# import os, tempfile
#
# hf_token = os.getenv('HF_TOKEN')
# if hf_token:
#     login(token=hf_token)
# else:
#     login()
#
# repo_name = 'your-username/jaguars_0226_segmented'
# api = HfApi(token=hf_token)
# api.create_repo(repo_id=repo_name, repo_type='dataset', private=True, exist_ok=True)
#
# ds_nodedup = fo.load_dataset(NON_DEDUP_DATASET)
# with tempfile.TemporaryDirectory(prefix='jid_hf_') as tmpdir:
#     export_dir = Path(tmpdir) / 'fiftyone_dataset'
#     ds_nodedup.export(export_dir=str(export_dir), dataset_type=fo.types.FiftyOneDataset, overwrite=True)
#     api.upload_large_folder(repo_id=repo_name, repo_type='dataset', folder_path=str(export_dir))
#     print(f'Uploaded to {repo_name}')

## Step 3: Setup Training Configuration

In [7]:
import copy
from jaguars.reidentification.experiments import get_default_config, ExperimentConfig
from jaguars.reidentification.training.train import run_processing as run_training
from jaguars.reidentification.wandb_results import fetch_latest_metrics_for_experiments

# Base config -- fixed for both runs
base_config = get_default_config()

# W&B
base_config.wandb.enabled = True
base_config.wandb.entity = 'jaguars'
base_config.wandb.project = 'camera-trap-reidentification'
base_config.wandb.tags = ['dedup_comparison']

# MiewID-MSv2 backbone
base_config.backbone.name = 'conservationxlabs/miewid-msv2'
base_config.backbone.pretrained = True
base_config.backbone.embedding_dim = 2152
base_config.backbone.input_size = 440

# ArcFace loss
base_config.training.loss_name = 'arcface'
base_config.model.arcface_margin = 0.5
base_config.model.arcface_scale = 64.0

# Training
base_config.training.num_epochs = 50
base_config.training.batch_size = 32
base_config.training.seed = 42

# FiftyOne
base_config.dataset.source = 'fiftyone'
base_config.dataset.fo_split_field = 'closed_set_split'
base_config.dataset.fo_label_field = 'ground_truth'
base_config.dataset.fo_patches_field = 'sam3_segmentations'

print('Base config ready')
print(f'  Backbone: {base_config.backbone.name}')
print(f'  Loss: {base_config.training.loss_name} (m={base_config.model.arcface_margin})')
print(f'  Epochs: {base_config.training.num_epochs}')

Base config ready
  Backbone: conservationxlabs/miewid-msv2
  Loss: arcface (m=0.5)
  Epochs: 50


In [9]:
# Define two experiment configs
experiments = []

# Experiment 1: Deduplicated (primary benchmark)
config_dedup = copy.deepcopy(base_config)
config_dedup.dataset.fo_dataset_name = DEDUP_DATASET
config_dedup.wandb.run_name = 'dedup_miewid_msv2_deduplicated'
config_dedup.wandb.tags = ['dedup_comparison', 'deduplicated']

experiments.append(ExperimentConfig(
    name='dedup_miewid_msv2_deduplicated',
    description='MiewID-MSv2 + ArcFace on deduplicated dataset',
    base_config=config_dedup,
    group='dedup_comparison',
    # tags=['dedup_comparison', 'deduplicated'],
))

# Experiment 2: Non-deduplicated (with duplicates)
config_nodedup = copy.deepcopy(base_config)
config_nodedup.dataset.fo_dataset_name = NON_DEDUP_DATASET
config_nodedup.wandb.run_name = 'dedup_miewid_msv2_with_duplicates'
config_nodedup.wandb.tags = ['dedup_comparison', 'with_duplicates']

experiments.append(ExperimentConfig(
    name='dedup_miewid_msv2_with_duplicates',
    description='MiewID-MSv2 + ArcFace on non-deduplicated dataset',
    base_config=config_nodedup,
    group='dedup_comparison',
    # tags=['dedup_comparison', 'with_duplicates'],
))

print(f'Defined {len(experiments)} experiments:')
for exp in experiments:
    print(f'  - {exp.name}: {exp.description}')

Defined 2 experiments:
  - dedup_miewid_msv2_deduplicated: MiewID-MSv2 + ArcFace on deduplicated dataset
  - dedup_miewid_msv2_with_duplicates: MiewID-MSv2 + ArcFace on non-deduplicated dataset


## Step 4: Run Experiments

In [ ]:
# Run all dedup comparison experiments
results = {}

for experiment in experiments:
    logger.info('=' * 60)
    logger.info('Running: %s', experiment.name)
    logger.info('  Description: %s', experiment.description)
    logger.info('  Dataset: %s', experiment.base_config.dataset.fo_dataset_name)
    logger.info('  Tags: %s', experiment.base_config.wandb.tags)
    logger.info('=' * 60)
    
    try:
        result = run_training(config=experiment.base_config)
        results[experiment.name] = result
        logger.info('Completed: %s', experiment.name)
        if hasattr(result, 'metrics'):
            logger.info('  Metrics: %s', result.metrics)
    except Exception as e:
        logger.error('Failed: %s -- %s', experiment.name, e)
        results[experiment.name] = {'error': str(e)}

print(f'Completed {len(results)} experiments')

19:05:05 - jid_logger.dedup_experiments - INFO - ============================================================
19:05:05 - jid_logger.dedup_experiments - INFO - Running: dedup_miewid_msv2_deduplicated
19:05:05 - jid_logger.dedup_experiments - INFO -   Description: MiewID-MSv2 + ArcFace on deduplicated dataset
19:05:05 - jid_logger.dedup_experiments - INFO -   Dataset: JID_HF_0226_Segmented_Deduplicated_Cached
19:05:05 - jid_logger.dedup_experiments - INFO -   Tags: ['dedup_comparison', 'deduplicated']
19:05:05 - jid_logger.dedup_experiments - INFO - ============================================================
19:05:05 - jid_logger.reidentification.training - INFO - Starting re-identification training...
19:05:05 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
19:05:05 - jid_logger.reidentification.training - INFO - Backbone: conservationxlabs/miewid-msv2
19:05:05 - jid_logger.reidentification.training - INFO - Device: cuda
19:05:05 - jid_logger.reidentification.t

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /sc/home/philipp.kolbe/.netrc.
wandb: Currently logged in as: hpi-philipp-kolbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


19:05:08 - jid_logger.reidentification.training - INFO - Loading dataset...
19:05:38 - jid_logger.reidentification.training - INFO - Dataset loaded:
19:05:38 - jid_logger.reidentification.training - INFO -   Train: 1632 samples
19:05:38 - jid_logger.reidentification.training - INFO -   Val: 167 samples
19:05:38 - jid_logger.reidentification.training - INFO -   Num classes: 175
19:05:38 - jid_logger.reidentification.training - INFO - Extracting embeddings with backbone...
Loading conservationxlabs/miewid-msv2 model via transformers AutoModel...
Building Model Backbone for efficientnetv2_rw_m model
config.model_name efficientnetv2_rw_m
model_name efficientnetv2_rw_m
final_in_features 2152


/sc/home/philipp.kolbe/conda3/envs/jid/lib/python3.14/site-packages/torch/nn/modules/module.py:2446: UserWarning: for conv_stem.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/sc/home/philipp.kolbe/conda3/envs/jid/lib/python3.14/site-packages/torch/nn/modules/module.py:2446: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/sc/home/philipp.kolbe/conda3/envs/jid/lib/python3.14/site-packages/torch/nn/modules/module.py:2446: UserWarning: for bn1.bias: copying from a non-meta parameter in the che

Loading weights:   0%|          | 0/1210 [00:00<?, ?it/s]

Model loaded successfully
  Backend: transformers.AutoModel (trust_remote_code=True)
  Parameters: 51,109,277
  Embedding dimension: 2152


Train embeddings:  65%|██████▍   | 33/51 [00:47<00:25,  1.43s/it]

## Step 5: Results Summary & Comparison

In [ ]:
results

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Load latest metrics from W&B for both dedup experiments
wandb_results = fetch_latest_metrics_for_experiments(
    experiments=experiments,
    entity=base_config.wandb.entity,
    project=base_config.wandb.project,
    additional_tags=['dedup_comparison'],
)

dedup_metrics = wandb_results.get('dedup_miewid_msv2_deduplicated', {})
nodedup_metrics = wandb_results.get('dedup_miewid_msv2_with_duplicates', {})

if 'error' in dedup_metrics:
    print(f"Warning (deduplicated): {dedup_metrics['error']}")
if 'error' in nodedup_metrics:
    print(f"Warning (with duplicates): {nodedup_metrics['error']}")

# Extract metrics for comparison
metric_keys = ['map', 'cmc@1', 'cmc@5', 'identity_balanced_map', 'map_min_total_9']
metric_labels = ['mAP', 'CMC@1', 'CMC@5', 'ib-mAP', 'mAP@9+']

print('=' * 60)
print('Deduplication Impact Summary (Latest W&B Runs)')
print('=' * 60)
print(f'{"Metric":<15} {"Deduplicated":>15} {"With Duplicates":>15} {"Diff":>10}')
print('-' * 60)

for key, label in zip(metric_keys, metric_labels):
    d = dedup_metrics.get(key, float('nan'))
    n = nodedup_metrics.get(key, float('nan'))
    diff = d - n if isinstance(d, (int, float)) and isinstance(n, (int, float)) else float('nan')
    print(f'{label:<15} {d:>15.4f} {n:>15.4f} {diff:>+10.4f}')

In [ ]:
# Save results to JSON for report
output_dir = Path('data/results')
output_dir.mkdir(parents=True, exist_ok=True)

dedup_results = {
    'deduplicated': dedup_metrics if isinstance(dedup_metrics, dict) else {},
    'with_duplicates': nodedup_metrics if isinstance(nodedup_metrics, dict) else {},
}

with open(output_dir / 'dedup_comparison_results.json', 'w') as f:
    json.dump(dedup_results, f, indent=2, default=str)
print(f'Saved to {output_dir / "dedup_comparison_results.json"}')

In [ ]:
# Generate comparison bar chart
fig, ax = plt.subplots(figsize=(3.3, 2.5))  # ACL column width

x = np.arange(len(metric_labels))
width = 0.35

vals_dedup = [dedup_metrics.get(k, 0) for k in metric_keys]
vals_nodedup = [nodedup_metrics.get(k, 0) for k in metric_keys]

bars1 = ax.bar(x - width/2, vals_dedup, width, label='Deduplicated', color='#4C72B0')
bars2 = ax.bar(x + width/2, vals_nodedup, width, label='With duplicates', color='#DD8452')

ax.set_ylabel('Score', fontsize=8)
ax.set_title('Deduplication Impact on Re-ID Metrics', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=7, rotation=15)
ax.legend(fontsize=7)
ax.set_ylim(0, 1.0)
ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()

fig_dir = Path('data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached')
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'dedup_comparison.pdf', dpi=300, bbox_inches='tight')
fig.savefig(fig_dir / 'dedup_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved dedup_comparison.pdf/png')